# NG15 free-spectral CW analysis: plotting results

Loads the samples saved by `run_ng15_free_spectral.py` and makes plots.
Run that script first (e.g. in a tmux session):

```bash
cd NG15 && python run_ng15_free_spectral.py
```

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

SAVEFILE = 'quickCW_ng15_free_spectral_rn.h5'

with h5py.File(SAVEFILE, 'r') as f:
    print(list(f.keys()))
    par_names = [n.decode() for n in f['par_names'][:]]
    samples = f['samples_cold'][0, :, :]
    log_likelihood = f['log_likelihood'][:]

print(samples.shape[0], 'thinned samples,', len(par_names), 'parameters')

## Likelihood trace

Useful to judge burn-in; adjust `burnin` below before making posterior plots.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(log_likelihood[0, :])
ax.set_xlabel('iteration')
ax.set_ylabel(r'$\log L$ (cold chain)')
plt.show()

burnin = samples.shape[0] // 4  # discard the first quarter (adjust as needed)
chain = samples[burnin:, :]

## CW source parameters

In [ ]:
cw_pars = ['0_log10_fgw', '0_log10_h', '0_log10_mc', '0_cos_gwtheta', '0_gwphi', '0_cos_inc']
labels = [r'$\log_{10} f_{\rm GW}$', r'$\log_{10} h$', r'$\log_{10} \mathcal{M}_c$',
          r'$\cos\theta_{\rm GW}$', r'$\phi_{\rm GW}$', r'$\cos\iota$']

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, par, lab in zip(axes.flat, cw_pars, labels):
    ax.hist(chain[:, par_names.index(par)], bins=50, density=True, histtype='step')
    ax.set_xlabel(lab)
plt.tight_layout()
plt.show()

## GWB power-law posterior

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].hist(chain[:, par_names.index('gwb_log10_A')], bins=50, density=True, histtype='step')
axes[0].set_xlabel(r'$\log_{10} A_{\rm GWB}$')
axes[1].hist(chain[:, par_names.index('gwb_gamma')], bins=50, density=True, histtype='step')
axes[1].set_xlabel(r'$\gamma_{\rm GWB}$')
plt.tight_layout()
plt.show()

## Per-pulsar free-spectral red noise posteriors

Violin plots of the `log10_rho` posterior in each frequency bin. Change `psr_name` to look at other
pulsars (any of the 67 names appearing in `par_names`).

In [ ]:
import bz2, glob, pickle

# observation timespan to convert bin index to frequency
tmins, tmaxs = [], []
for psrfile in sorted(glob.glob('data/jar/*.bz2.pkl')):
    with bz2.BZ2File(psrfile, 'rb') as f:
        p = pickle.load(f)
        tmins.append(p.toas.min()); tmaxs.append(p.toas.max())
Tspan = np.max(tmaxs) - np.min(tmins)
print('Tspan = %.2f yr' % (Tspan / 3.15576e7))

In [ ]:
psr_name = 'J1713+0747'
idx_rho = [i for i, n in enumerate(par_names) if n.startswith(psr_name + '_red_noise_log10_rho')]
freqs = np.arange(1, len(idx_rho) + 1) / Tspan

fig, ax = plt.subplots(figsize=(8, 4))
ax.violinplot([chain[:, i] for i in idx_rho], positions=np.log10(freqs), widths=0.04)
ax.set_xlabel(r'$\log_{10}(f/{\rm Hz})$')
ax.set_ylabel(r'$\log_{10}\rho$')
ax.set_title(f'{psr_name} free-spectral red noise posterior')
plt.show()

In [ ]:
# median log10_rho across bins for all pulsars at a glance
psr_names = sorted({n.split('_red_noise')[0] for n in par_names if '_red_noise_log10_rho' in n})
med = np.array([[np.median(chain[:, par_names.index(f'{p}_red_noise_log10_rho_{k}')])
                 for k in range(len(idx_rho))] for p in psr_names])

fig, ax = plt.subplots(figsize=(10, 12))
im = ax.imshow(med, aspect='auto', cmap='viridis')
ax.set_xlabel('frequency bin')
ax.set_yticks(np.arange(len(psr_names)), labels=psr_names, fontsize=6)
fig.colorbar(im, label=r'median $\log_{10}\rho$')
plt.show()